# Full Phugoid Model


In [Lesson 1](./01-theory.ipynb), we developed an idealized model of phugoid motion with no drag. In [Lesson 2](./02-oscillation.ipynb), we studied small perturbations around trimmed flight (straight-line phugoid), leading to simple harmonic motion. A useful pattern of re-writing a second-order differential equation as a system of two first-order equations allowed us to use Euler's method to compute a two-component state. We learned about convergence and calculated the error of the numerical solution, comparing with an analytical solution. That is a good foundation!

We now return to the full dynamical model, include drag, and allow the aircraft speed and trajectory angle to vary together.

The numerical method stays the same, but the model and state variables change:

| Aspect | Lesson 2 | Lesson 3 |
|---|---|---|
| Model | Linearized oscillation | Nonlinear, damped phugoid |
| State | $u=[z,b]$ | $u=[v,\theta,x,y]$ |
| Vertical coordinate | $z$: depth, positive downward | $y$: altitude, positive upward |
| Right-hand side | `rhs_linear_phugoid()` | `rhs_full_phugoid()` |
| Numerical method | Forward Euler | The same Forward Euler step |

The change from downward-positive depth to upward-positive altitude deserves particular attention. Here, $y$ is a spatial coordinate used to draw the aircraft's trajectory in the sky, so increasing $y$ means climbing. We use $\theta$ for the trajectory angle and take it as positive above the horizontal.

```{figure} ./figures/glider_forces-lesson3.png
:label: fig-full-phugoid-forces
:alt: Lift, drag, and weight acting on a glider whose trajectory angle is positive above the horizontal
:align: center

Forces on a glider with a positive trajectory angle.
```


:::{warning .simple .dropdown icon=false open=false} On paper
Follow along the derivation of the mathematical model and keep those notes handy when you begin reconstructing the code for the numerical solution. This will help you not only create a base of understanding but create awareness to spot things like a pesky sign error.
:::

In [](#fig-full-phugoid-forces), $L$ is lift, $W=mg$ is weight, $D$ is drag, and $\theta$ is the instantaneous trajectory angle. Resolving Newton's second law parallel and perpendicular to the trajectory gives

$$
\label{eq-full-phugoid-force-balance}
\begin{aligned}
m\frac{dv}{dt} &= -W\sin\theta-D,\\
mv\frac{d\theta}{dt} &= -W\cos\theta+L.
\end{aligned}
$$

On the direction normal to the trajectory, we used that the aircraft travels a small distance $ds=Rd\theta$ along the curved trajectory, at a speed $v=ds/dt=R \, d\theta/dt$. This helps us write the centripetal acceleration $v^2/R$ as $v\, d\theta/dt$.

Dividing [Equation %s](#eq-full-phugoid-force-balance) by the weight and adopting primes for the time derivative gives

$$
\label{eq-full-phugoid-normalized-balance}
\begin{aligned}
\frac{v'}{g} &= -\sin\theta-\frac{D}{W},\\
\frac{v}{g}\theta' &= -\cos\theta+\frac{L}{W}.
\end{aligned}
$$


From [Lesson 1](./01-theory.ipynb), the lift-to-weight ratio is written in terms of the trim speed, $L/W=v^2/v_t^2$. Lift and drag share the same dynamic-pressure factor:

$$
\label{eq-full-phugoid-aerodynamic-forces}
L=C_LS\frac{1}{2}\rho v^2,
\qquad
D=C_DS\frac{1}{2}\rho v^2.
$$

It follows from [Equation %s](#eq-full-phugoid-aerodynamic-forces) that $D/L=C_D/C_L$. Substituting these relations into [Equation %s](#eq-full-phugoid-normalized-balance) produces the nonlinear velocity-and-angle model:

$$
\label{eq-full-phugoid-dynamics}
\begin{aligned}
v' &=
-g\sin\theta
-\frac{C_D}{C_L}\frac{g}{v_t^2}v^2,\\
\theta' &=
-\frac{g}{v}\cos\theta
+\frac{g}{v_t^2}v.
\end{aligned}
$$

The factor $C_D/C_L$ is the inverse of the aerodynamic efficiency $L/D$. It supplies damping: when the drag coefficient is zero, the damping term disappears. More aerodynamically efficient aircraft therefore have a more weakly damped phugoid mode.


## The initial value problem


To draw the flight path, we also integrate the spatial coordinates. The horizontal position $x$ and upward-positive altitude $y$ satisfy

$$
\label{eq-full-phugoid-kinematics}
\begin{aligned}
x'(t) &= v\cos\theta,\\
y'(t) &= v\sin\theta.
\end{aligned}
$$

Together, [Equation %s](#eq-full-phugoid-dynamics) and [Equation %s](#eq-full-phugoid-kinematics) form a system of four first-order differential equations. We need one initial value for every state variable:

$$
\label{eq-full-phugoid-initial-conditions}
v(0)=v_0,\qquad
\theta(0)=\theta_0,\qquad
x(0)=x_0,\qquad
y(0)=y_0.
$$


## Solve with Forward Euler


:::{warning .simple .dropdown icon=false open=false} In your notebook

Reconstruct the model, functions, time integration, plots, and convergence study in your own notebook. The worked code deliberately keeps `euler_step()` visible: compare it with the refactoring you completed in [Lesson 2](./02-oscillation.ipynb), and identify which part represents the numerical method and which part represents the new physical model.
:::

Forward Euler replaces a time derivative by a forward difference. For the speed,

$$
\label{eq-full-phugoid-forward-difference}
v'(t_n)\approx\frac{v^{n+1}-v^n}{\Delta t},
$$

where the superscript $n$ identifies the state at time $t_n$. Using the form of [Equation %s](#eq-full-phugoid-forward-difference) for each of the differential equations and solving for the next state gives

$$
\label{eq-full-phugoid-euler-system}
\begin{aligned}
v^{n+1} &= v^n+\Delta t\left(
-g\sin\theta^n
-\frac{C_D}{C_L}\frac{g}{v_t^2}(v^n)^2
\right),\\
\theta^{n+1} &= \theta^n+\Delta t\left(
-\frac{g}{v^n}\cos\theta^n
+\frac{g}{v_t^2}v^n
\right),\\
x^{n+1} &= x^n+\Delta t\,v^n\cos\theta^n,\\
y^{n+1} &= y^n+\Delta t\,v^n\sin\theta^n.
\end{aligned}
$$

Every right-hand side in [Equation %s](#eq-full-phugoid-euler-system) uses the same old state. After evaluating all four derivatives, we advance the complete state together.


We can collect the state and its derivatives into vectors:

$$
\label{eq-full-phugoid-vector-system}
u=
\begin{bmatrix}
v\\ \theta\\ x\\ y
\end{bmatrix},
\qquad
u'=f(u)=
\begin{bmatrix}
-g\sin\theta-\dfrac{C_D}{C_L}\dfrac{g}{v_t^2}v^2\\
-\dfrac{g}{v}\cos\theta+\dfrac{g}{v_t^2}v\\
v\cos\theta\\
v\sin\theta
\end{bmatrix}.
$$

In vector form, one Forward Euler step is simply

$$
\label{eq-forward-euler-vector-full-model}
u^{n+1}=u^n+\Delta t\,f(u^n).
$$

We will adopt the modular approach from [Lesson 2](./02-oscillation.ipynb) after the agent-driven refactoring, where we defined a Python function for the system dynamics and another for the time stepping. This is the key transition: `rhs_full_phugoid()` represents the model with drag and returns four derivatives, but `euler_step()` still implements [Equation %s](#eq-forward-euler-vector-full-model) without needing to know the length or meaning of the state vector. That is the power of array operations!

We begin by importing Python's array and plotting libraries using the conventional aliases: `np` for NumPy and `plt` for Matplotlib.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

Next, we need to choose representative model parameters and initial conditions. Angles in the state are measured in radians because `np.sin()` and `np.cos()` expect radians. The ratio $C_L/C_D=40$ represents a reasonably efficient sailplane, and $v_t=30\ \mathrm{m/s}$ is a representative trim speed. These are exploratory values rather than a model of a particular aircraft.


In [ ]:
# Model parameters.
g = 9.81  # gravitational acceleration (m/s**2)
v_t = 30.0  # trim speed (m/s)
C_D = 1.0 / 40.0  # drag coefficient
C_L = 1.0  # lift coefficient

# Initial conditions.
v_0 = v_t  # speed (m/s)
theta_0 = 0.0  # trajectory angle (rad)
x_0 = 0.0  # horizontal position (m)
y_0 = 1000.0  # altitude (m)


The function below translates [Equation %s](#eq-full-phugoid-vector-system) directly into code. The name `rhs_full_phugoid` distinguishes this nonlinear four-state model from the linear two-state function in the previous lesson. We're also providing a detailed description of the function parameters and return values in the _docstring_: the commented block under the function name. This block is shown to the user when appending a question mark (`?`) to the function name, or using the built-in `help()` functionality.

In [ ]:
def rhs_full_phugoid(u, C_L, C_D, g, v_t):
    '''Return the derivatives for the nonlinear, damped phugoid model.

    Parameters
    ----------
    u : np.ndarray
        State vector [v, theta, x, y].
    C_L : float
        Lift coefficient.
    C_D : float
        Drag coefficient.
    g : float
        Gravitational acceleration.
    v_t : float
        Trim speed.

    Returns
    -------
    np.ndarray
        Derivative vector [dv/dt, dtheta/dt, dx/dt, dy/dt].
    '''
    v, theta, x, y = u
    return np.array([
        -g * np.sin(theta) - (C_D / C_L) * (g / v_t**2) * v**2,
        -(g / v) * np.cos(theta) + (g / v_t**2) * v,
        v * np.cos(theta),
        v * np.sin(theta),
    ])


Compare each returned array entry with the corresponding row of [Equation %s](#eq-full-phugoid-vector-system). The position values $x$ and $y$ do not appear on the right-hand side, but keeping them in the state lets the same integration loop advance both the flight dynamics and the trajectory.

The Forward Euler function has the same vector update as in the previous lesson. This version uses `*args` so it can forward the larger model-parameter list without naming those parameters.

In [ ]:
def euler_step(u, f, dt, *args):
    '''Return the next state using one Forward Euler step.

    Parameters
    ----------
    u : np.ndarray
        State at the current time.
    f : callable
        Function that returns the state derivatives.
    dt : float
        Time-step size.
    *args
        Additional positional arguments passed to f.

    Returns
    -------
    np.ndarray
        State after one Forward Euler step.
    '''
    return u + dt * f(u, *args)
    

:::{note} Python refresher — forwarding arguments with `*args`
:icon: false

In a function definition, `*args` collects any additional positional arguments into a tuple.  The asterisk `*` is an operator that tells Python to bundle all remaining arguments together, while `args` is just a naming convention. 
The expression `f(u, *args)` expands that tuple when calling `f`. For example,

```python
euler_step(u, rhs_full_phugoid, dt, C_L, C_D, g, v_t)
```

makes `args` equal to `(C_L, C_D, g, v_t)` inside `euler_step()`, so its call to `f` is equivalent to `rhs_full_phugoid(u, C_L, C_D, g, v_t)`. The Euler function remains independent of the particular model.
(Read the Python documentation about [Arbitrary Argument Lists](https://docs.python.org/3/tutorial/controlflow.html#arbitrary-argument-lists) for more details.)
:::

Now choose the final time $T$ and step size $\Delta t$. The variable `num_steps` counts updates, while the time grid contains `num_steps + 1` points because it includes both endpoints. `np.empty()` allocates a two-dimensional state-history array with one row per time step and four columns ordered as $[v,\theta,x,y]$.
In the `for`-loop, the function `euler_step(` is called to get the solution at time step `n+1`.

In [ ]:
T = 100.0  # length of the time interval (s)
dt = 0.1   # time-step size (s)
num_steps = int(round(T / dt))
t = np.linspace(0.0, T, num=num_steps + 1)

# Store [v, theta, x, y] at every time step.
u_history = np.empty((num_steps + 1, 4))
u_history[0] = np.array([v_0, theta_0, x_0, y_0])

for n in range(num_steps):
    u_history[n + 1] = euler_step(
        u_history[n], rhs_full_phugoid, dt, C_L, C_D, g, v_t
    )


## Plot the trajectory


The state history contains everything we need to plot the trajectory of the glider. We extract the position coordinates using array indexing. A colon in the row position selects every time point, while columns 2 and 3 contain horizontal position and altitude, respectively.

:::[aside}
If you need a quick and simple review of indexing in Python, you may find this video by Prof. Barba useful: https://youtu.be/R3eNdpWHKOs
:::

In [ ]:
x = u_history[:, 2]
y = u_history[:, 3]


In [ ]:
# Set the font family and size to use for Matplotlib figures.
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 12

fig, ax = plt.subplots(figsize=(8,3))

ax.set_title(f'Glider trajectory over {T:g} s')
ax.set_xlabel('Horizontal position, x (m)')
ax.set_ylabel('Altitude, y (m)')
ax.grid()
ax.plot(x, y, color='tab:red', linestyle='-', linewidth=2)
fig.tight_layout()


## Grid convergence


In [Lesson 2](./02-oscillation.ipynb), when we studied the straight-line phugoid under a small perturbation, we looked at convergence by comparing the numerical solution with the exact solution. But we do not have an exact solution for this full nonlinear model. Instead, we compare solutions computed with different time-step sizes against the solution on a fine grid.

The finest-grid result is a **numerical reference**, not an exact solution. The resulting differences provide evidence of grid convergence only if the reference grid is sufficiently resolved and all compared grids represent the same initial-value problem over the same interval.

Compute a state history for each time-step size:


In [ ]:
dt_values = [0.1, 0.05, 0.01, 0.005, 0.001]
u_histories = [] # an empty list for the solution on each grid

for dt_trial in dt_values:
    num_steps_trial = int(round(T / dt_trial))
    u_trial = np.empty((num_steps_trial + 1, 4))
    u_trial[0] = np.array([v_0, theta_0, x_0, y_0])

    for n in range(num_steps_trial):
        u_trial[n + 1] = euler_step(
            u_trial[n],
            rhs_full_phugoid,
            dt_trial,
            C_L,
            C_D,
            g,
            v_t,
        )

    u_histories.append(u_trial)


To compare arrays sampled on different grids, their time points must align. For nested grids, the ratio $\Delta t_{\mathrm{coarse}}/\Delta t_{\mathrm{fine}}$ is an integer. A slice such as `q_fine[::refinement_ratio]` then selects the fine-grid values at every coarse-grid time.

The function below derives that ratio from the two step sizes. `np.isclose()` checks the floating-point relation before slicing, and the shape check confirms that the endpoints and sample counts also align. Its generic names `q_coarse` and `q_fine` make clear that it compares one sampled quantity; below, that quantity will be horizontal position.


In [ ]:
def discrete_l1_difference(
    q_coarse, q_fine, dt_coarse, dt_fine
):
    '''Return a discrete L1 difference on two nested time grids.'''
    refinement_ratio = int(round(dt_coarse / dt_fine))

    if not np.isclose(
        dt_coarse, refinement_ratio * dt_fine
    ):
        raise ValueError('Time-step sizes do not define nested grids.')

    q_fine_on_coarse_grid = q_fine[::refinement_ratio]

    if q_fine_on_coarse_grid.shape != q_coarse.shape:
        raise ValueError('Grid endpoints or sample counts do not align.')

    return dt_coarse * np.sum(
        np.abs(q_coarse - q_fine_on_coarse_grid)
    )


Use the horizontal-position history from the finest grid as the numerical reference, then calculate the discrete $L_1$ difference for every coarser solution. Negative indexing selects the final item in each list.


In [ ]:
dt_reference = dt_values[-1]
x_reference = u_histories[-1][:, 2]
difference_values = []

for dt_trial, u_trial in zip(
    dt_values[:-1], u_histories[:-1]
):
    x_trial = u_trial[:, 2]
    difference_values.append(
        discrete_l1_difference(
            x_trial,
            x_reference,
            dt_trial,
            dt_reference,
        )
    )


In [ ]:
dt_array = np.array(dt_values[:-1])

fig, ax = plt.subplots(figsize=(5.0, 5.0))
ax.set_title(r'$L_1$ difference vs. time-step size')
ax.set_xlabel(r'$\Delta t$ (s)')
ax.set_ylabel(r'$L_1$ difference in $x$')
ax.grid()
ax.loglog(
    dt_array,
    difference_values,
    color='tab:blue',
    linestyle='--',
    marker='o',
)
ax.set_aspect('equal', adjustable='box')
fig.tight_layout()


As the time step decreases, the difference from the finest-grid reference also decreases. This is useful numerical evidence, but it does not prove that the finest reference is exact or that every important feature has been resolved.


### Observed order of convergence


Suppose three step sizes have a constant refinement ratio $r$, with $f_1$ the finest-grid result and $f_3$ the coarsest. Let $D_{21}$ be a norm of the difference between the medium and fine results, and $D_{32}$ the corresponding difference between the coarse and medium results. The observed order is

$$
\label{eq-observed-order-full-phugoid}
p=
\frac{\log(D_{32}/D_{21})}{\log r}.
$$

For a first-order method in its asymptotic range, we expect $D_{32}/D_{21}\approx r$ and therefore $p\approx1$. We use three nested grids with $r=2$.


In [ ]:
refinement_ratio = 2
dt_fine = 0.001
dt_order = [
    dt_fine,
    refinement_ratio * dt_fine,
    refinement_ratio**2 * dt_fine,
]
u_order = []

for dt_trial in dt_order:
    num_steps_trial = int(round(T / dt_trial))
    u_trial = np.empty((num_steps_trial + 1, 4))
    u_trial[0] = np.array([v_0, theta_0, x_0, y_0])

    for n in range(num_steps_trial):
        u_trial[n + 1] = euler_step(
            u_trial[n],
            rhs_full_phugoid,
            dt_trial,
            C_L,
            C_D,
            g,
            v_t,
        )

    u_order.append(u_trial)

difference_medium_fine = discrete_l1_difference(
    u_order[1][:, 2],
    u_order[0][:, 2],
    dt_order[1],
    dt_order[0],
)
difference_coarse_medium = discrete_l1_difference(
    u_order[2][:, 2],
    u_order[1][:, 2],
    dt_order[2],
    dt_order[1],
)

observed_order = (
    np.log(difference_coarse_medium / difference_medium_fine)
    / np.log(refinement_ratio)
)
print(f'Observed order of convergence: p = {observed_order:.3f}')


The computed value is close to 1, in agreement with the first-order convergence expected for Forward Euler. This result is stronger than a single trajectory plot: it checks how differences change under systematic refinement. It remains conditional on using nested grids, reaching the asymptotic refinement range, and having a sufficiently resolved fine-grid reference.


## Paper-airplane challenge


:::{warning .simple .dropdown icon=false open=false} In your notebook

Suppose you want to use the phugoid model to improve the flight distance of a paper airplane. For a fixed lift-to-drag ratio, investigate which initial speed and launch angle carry the airplane farthest from a given height.

- Assume $L/D=5$, a value motivated by measurements reported by [@feng2009].
- Use a trim speed of $4.9\ \mathrm{m/s}$.
- Search for a combination of launch angle and speed that maximizes horizontal distance.
- Decide how the calculation should detect the moment the airplane reaches the ground.
- Explain how you would check whether the result is physically and numerically credible.

State the ranges you searched, the stopping rule, and the evidence supporting your answer. This challenge is adapted from the computational phugoid exercise of [@simanca2002].
:::
